In [3]:
import os
import glob
import optuna
import joblib
import datetime
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import yaml

from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ============================================================
# 1. Load Configuration
# ============================================================
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)

AEZ_CSV_PATH  = config['paths']['aez_data_dir']
OUTPUT_FOLDER = config['paths']['models_joblib_dir']
PLOTS_DIR     = config['paths']['plots_dir']
STATS_FOLDER  = config['paths']['stats_dir']

N_ESTIMATORS     = config['parameters']['rf_n_estimators']
MAX_DEPTH        = config['parameters']['rf_max_depth']
TARGET_AEZS      = config['parameters']['target_aezs']
TARGET_NUTRIENTS = config['parameters']['target_nutrients']

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(STATS_FOLDER,  exist_ok=True)

# ============================================================
# 2. Constants
# ============================================================

# Bins defined by Soil Health Portal (Govt. of India)
bin_dict = {
    "N"  : [0, 280, 560, np.inf],
    "P"  : [0, 10, 25, np.inf],
    "K"  : [0, 120, 280, np.inf],
    "OC" : [0.0, 0.50, 0.75, 1.0]
}

selected_feature_names = [
    'temp',
    'SI',
    'RI',
    'precipitation',
    'sand05',
    'silt05',
    'sand515',
    'TGSI',
    'NDVI_Kharif', 'NDVI_Rabi', 'NDVI_Zaid',
    'pH_0-5', 'pH_5-15',
    'NIRv_Kharif', 'NIRv_Rabi', 'NIRv_Zaid'
]

# ============================================================
# 3. Helper Functions
# ============================================================

def add_indices(df):
    """Compute and append spectral/soil indices to dataframe."""
    df["NDVI"]  = (df["NIR"] - df["RED"])   / (df["NIR"] + df["RED"])
    df["GNDVI"] = (df["NIR"] - df["GREEN"]) / (df["NIR"] + df["GREEN"])
    df["SAVI"]  = ((df["NIR"] - df["RED"])  / (df["NIR"] + df["RED"] + 0.5)) * 1.5
    df["NDWI"]  = (df["GREEN"] - df["NIR"]) / (df["GREEN"] + df["NIR"])
    df["BI"]    = np.sqrt((df["RED"]**2 + df["GREEN"]**2 + df["BLUE"]**2) / 3)
    df["SI"]    = (df["RED"]   - df["BLUE"])  / (df["RED"]   + df["BLUE"])
    df["HI"]    = (2 * df["RED"] - df["GREEN"] - df["BLUE"]) / (df["GREEN"] - df["BLUE"])
    df["CI"]    = (df["RED"]   - df["GREEN"]) / (df["RED"]   + df["GREEN"])
    df["RI"]    = (df["RED"]**2) / (df["BLUE"] * df["GREEN"]**3)
    df["TGSI"]  = (df["SWIR1"] - df["NIR"])  / (df["SWIR1"] + df["NIR"])
    df["NCI"]   = (df["SWIR1"] - df["SWIR2"]) / (df["SWIR1"] + df["SWIR2"])
    df["EVI"]   = (2.5 * (df["NIR"] - df["RED"])) / (df["NIR"] + 6*df["RED"] - 7.5*df["BLUE"] + 1)
    return df


def smape(y_true, y_pred):
    """Symmetric Mean Absolute Percentage Error."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    return np.mean(np.abs(y_true - y_pred) / (denominator + 1e-6)) * 100


def compute_metrics(y_true, y_pred):
    """Return dict of R2, RMSE, MAE, sMAPE — computed once, used everywhere."""
    return {
        "R2"    : round(r2_score(y_true, y_pred), 4),
        "RMSE"  : round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
        "MAE"   : round(mean_absolute_error(y_true, y_pred), 4),
        "sMAPE" : round(smape(y_true, y_pred), 4)
    }


def create_objective(X_train_data, y_train_data, features, n_est, max_d):
    """
    Closure that creates an Optuna objective with fully localised scope.
    Prevents variable leakage between AEZ/nutrient iterations.
    """
    def objective(trial):
        params = {
            'n_estimators'    : n_est,
            'max_depth'       : max_d,
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
            'min_samples_leaf' : trial.suggest_int('min_samples_leaf',  2, 10),
            'max_features'    : trial.suggest_float('max_features', 0.3, 1.0),
            'random_state'    : 42,
            'n_jobs'          : -1
        }
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_data, y_train_data, test_size=0.2, random_state=42
        )
        bin_counts = X_tr['bin'].value_counts()
        weights = X_tr['bin'].map(
            lambda b: 10000 * 1.0 / np.log(bin_counts[b] + 1)
        )
        model = RandomForestRegressor(**params)
        model.fit(X_tr[features], y_tr, sample_weight=weights)
        y_pred = model.predict(X_val[features])
        return r2_score(y_val, y_pred)
    return objective


def save_scatter_plot(y_true, y_pred, title, save_path):
    """True vs Predicted scatter — saves to disk, never blocks execution."""
    plt.figure(figsize=(8, 5))
    plt.scatter(y_true, y_pred, alpha=0.5, color='steelblue', s=15)
    plt.plot(
        [y_true.min(), y_true.max()],
        [y_true.min(), y_true.max()],
        'k--', lw=2, label='Ideal'
    )
    plt.xlabel('True Values')
    plt.ylabel('Predicted Values')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()  # CRITICAL: never use plt.show() in automation


# ============================================================
# 4. Master Execution Loop
# ============================================================
master_stats_list = []
failed_list       = []
total_models      = len(TARGET_AEZS) * len(TARGET_NUTRIENTS)
model_counter     = 0

for aez in TARGET_AEZS:

    print(f"\n{'='*60}")
    print(f"  PROCESSING AEZ: {aez}")
    print(f"{'='*60}")

    # ── Check file exists before doing anything ───────────────
    file_path = os.path.join(AEZ_CSV_PATH, f"AEZ_{aez}.csv")
    if not os.path.exists(file_path):
        print(f"  ✗ Skipping AEZ {aez}: file not found at {file_path}")
        for nutrient in TARGET_NUTRIENTS:
            failed_list.append({
                "AEZ": aez, "Nutrient": nutrient,
                "Error": "CSV file not found"
            })
        continue

    # ── Load and preprocess ONCE per AEZ ─────────────────────
    df_raw = pd.read_csv(file_path)
    df_raw = df_raw.drop(
        columns=['district', 'village', 'date', 'start_date', 'end_date', 'ae_regcode'],
        errors='ignore'
    )
    df_raw = add_indices(df_raw)
    df_raw = df_raw.drop(
        columns=['RED', 'BLUE', 'GREEN', 'SWIR1', 'SWIR2', 'NIR'],
        errors='ignore'
    )
    df_raw = df_raw.replace([np.inf, -np.inf], np.nan).dropna()
    df_raw[df_raw.select_dtypes(np.float64).columns] = \
        df_raw.select_dtypes(np.float64).astype(np.float32)
    df_raw[df_raw.select_dtypes(np.int64).columns] = \
        df_raw.select_dtypes(np.int64).astype(np.int16)

    print(f"  Loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")

    # ── Inner loop: one model per nutrient ────────────────────
    for nutrient in TARGET_NUTRIENTS:

        model_counter += 1
        print(f"\n  [{model_counter}/{total_models}]  AEZ {aez}  |  {nutrient}")
        print(f"  {'-'*40}")

        try:

            # ── Plot directory ────────────────────────────────
            PLOT_SAVE_PATH = os.path.join(PLOTS_DIR, f"AEZ_{aez}", nutrient)
            os.makedirs(PLOT_SAVE_PATH, exist_ok=True)

            # ── Work on a fresh copy for this nutrient ────────
            df = df_raw.copy()

            # ── Dynamic binning ───────────────────────────────
            base_bins = bin_dict[nutrient]
            df['base_bin'] = pd.cut(
                df[nutrient], bins=base_bins,
                labels=["Low", "Mid", "High"], include_lowest=True
            )

            target_samples_per_bin = int(np.ceil(len(df) / 4))
            custom_bins = []

            for i, label in enumerate(["Low", "Mid", "High"]):
                subset = df[df['base_bin'] == label]
                if not subset.empty:
                    n_points = len(subset)
                    sub_bins = max(1, int(np.ceil(n_points / target_samples_per_bin)))
                    edges = np.linspace(
                        subset[nutrient].min(),
                        subset[nutrient].max(),
                        sub_bins + 1
                    )
                    if i > 0:
                        edges = edges[1:]  # avoid duplicate boundary
                    custom_bins.extend(edges)

            custom_bins = sorted(set(custom_bins))
            df['bin'] = pd.cut(df[nutrient], bins=custom_bins, include_lowest=True)

            # ── Binning histogram ─────────────────────────────
            plt.figure(figsize=(10, 6))
            counts, bin_edges, _ = plt.hist(
                df[nutrient], bins=custom_bins, edgecolor='black', alpha=0.7
            )
            for i in range(len(counts)):
                if counts[i] > 0:
                    plt.text(
                        (bin_edges[i] + bin_edges[i+1]) / 2, counts[i],
                        str(int(counts[i])),
                        ha='center', va='bottom', fontsize=8, rotation=45
                    )
            plt.xlabel(f'{nutrient} Value')
            plt.ylabel('Frequency')
            plt.title(f'Histogram of {nutrient} — AEZ {aez}')
            plt.grid(True)
            plt.tight_layout()
            plt.savefig(os.path.join(PLOT_SAVE_PATH, "binning.png"), dpi=150)
            plt.close()

            # ── Train / Test split ────────────────────────────
            X = df
            y = df[nutrient]
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42
            )

            # ── Optuna hyperparameter search ──────────────────
            objective = create_objective(
                X_train, y_train, selected_feature_names,
                N_ESTIMATORS, MAX_DEPTH
            )
            study = optuna.create_study(direction='maximize')
            optuna.logging.set_verbosity(optuna.logging.WARNING)
            study.optimize(objective, n_trials=26, timeout=1300)

            MIN_SAMPLES_SPLIT = study.best_trial.params['min_samples_split']
            MIN_SAMPLES_LEAF  = study.best_trial.params['min_samples_leaf']
            MAX_FEATURES      = study.best_trial.params['max_features']

            print(f"  Optuna best R²: {study.best_value:.4f}")
            print(f"  Params: split={MIN_SAMPLES_SPLIT}, "
                  f"leaf={MIN_SAMPLES_LEAF}, features={MAX_FEATURES:.3f}")

            # ── 10-Fold Cross Validation ──────────────────────
            kf = KFold(n_splits=10, shuffle=True, random_state=42)
            models             = []
            r2_scores          = []
            scores             = []
            train_test_split_idx = []

            for fold_num, (train_idx, test_idx) in enumerate(kf.split(X_train)):

                X_val_train = X_train.iloc[train_idx]
                X_val_test  = X_train.iloc[test_idx]
                y_val_train = y_train.iloc[train_idx]
                y_val_test  = y_train.iloc[test_idx]

                bin_counts = X_val_train['bin'].value_counts()
                weights = X_val_train['bin'].map(
                    lambda b: 10000 * 1.0 / bin_counts[b]
                )
                train_test_split_idx.append((train_idx, test_idx))

                rfr = RandomForestRegressor(
                    n_estimators    = N_ESTIMATORS,
                    max_depth       = MAX_DEPTH,
                    min_samples_split = MIN_SAMPLES_SPLIT,
                    min_samples_leaf  = MIN_SAMPLES_LEAF,
                    max_features    = MAX_FEATURES,
                    random_state    = 42,
                    n_jobs          = -1
                )
                rfr.fit(
                    X_val_train[selected_feature_names],
                    y_val_train,
                    sample_weight=weights
                )
                models.append(rfr)

                y_pred_fold = rfr.predict(X_val_test[selected_feature_names])

                # compute once, use twice — no redundant calls
                fold_metrics = compute_metrics(y_val_test, y_pred_fold)
                r2_scores.append(fold_metrics["R2"])
                scores.append(fold_metrics)

                print(f"    Fold {fold_num+1:>2}  "
                      f"R²={fold_metrics['R2']:.4f}  "
                      f"RMSE={fold_metrics['RMSE']:.4f}  "
                      f"MAE={fold_metrics['MAE']:.4f}  "
                      f"sMAPE={fold_metrics['sMAPE']:.4f}")

            print(f"  CV Mean R²: {np.mean(r2_scores):.4f}")

            # ── Extract best fold model ───────────────────────
            best_model_idx = np.argmax(r2_scores)
            best_model     = models[best_model_idx]
            train_idx, test_idx = train_test_split_idx[best_model_idx]

            X_val_train = X_train.iloc[train_idx]
            X_val_test  = X_train.iloc[test_idx]
            y_val_train = y_train.iloc[train_idx]
            y_val_test  = y_train.iloc[test_idx]

            print(f"  Best fold: {best_model_idx+1}  "
                  f"R²={r2_scores[best_model_idx]:.4f}")

            # ── Evaluate on all three sets ────────────────────
            y_pred_cal_train = best_model.predict(X_val_train[selected_feature_names])
            y_pred_cal_test  = best_model.predict(X_val_test[selected_feature_names])
            y_pred_test      = best_model.predict(X_test[selected_feature_names])

            m_cal_train = compute_metrics(y_val_train, y_pred_cal_train)
            m_cal_test  = compute_metrics(y_val_test,  y_pred_cal_test)
            m_test      = compute_metrics(y_test,      y_pred_test)

            print(f"\n  CAL_TRAIN  → R²={m_cal_train['R2']}  "
                  f"RMSE={m_cal_train['RMSE']}  MAE={m_cal_train['MAE']}")
            print(f"  CAL_TEST   → R²={m_cal_test['R2']}   "
                  f"RMSE={m_cal_test['RMSE']}  MAE={m_cal_test['MAE']}")
            print(f"  TEST       → R²={m_test['R2']}       "
                  f"RMSE={m_test['RMSE']}  MAE={m_test['MAE']}")

            # ── Per-AEZ per-nutrient stats CSV ────────────────
            # Preserves full detail: CAL_TRAIN, CAL_TEST, TEST, CROSS_VAL_MEAN
            cross_val_mean = pd.DataFrame(scores).mean()

            stats_df = pd.DataFrame([
                {"AEZ": aez, "Set": "CAL_TRAIN",      "Property": nutrient, **m_cal_train},
                {"AEZ": aez, "Set": "CAL_TEST",        "Property": nutrient, **m_cal_test},
                {"AEZ": aez, "Set": "TEST",             "Property": nutrient, **m_test},
                {"AEZ": aez, "Set": "CROSS_VAL_MEAN",  "Property": nutrient,
                 "R2"    : round(cross_val_mean["R2"],    4),
                 "RMSE"  : round(cross_val_mean["RMSE"],  4),
                 "MAE"   : round(cross_val_mean["MAE"],   4),
                 "sMAPE" : round(cross_val_mean["sMAPE"], 4)}
            ])

            STATS_PATH = os.path.join(STATS_FOLDER, f"AEZ_{aez}")
            os.makedirs(STATS_PATH, exist_ok=True)
            stats_df.to_csv(
                os.path.join(STATS_PATH, f"stats_{nutrient}.csv"),
                index=False
            )

            # ── True vs Predicted plots ───────────────────────
            for set_name, y_true, y_pred_vals in [
                ("CAL_TRAIN", y_val_train, y_pred_cal_train),
                ("CAL_TEST",  y_val_test,  y_pred_cal_test),
                ("TEST",      y_test,      y_pred_test)
            ]:
                save_scatter_plot(
                    y_true, y_pred_vals,
                    title=f"{nutrient} — AEZ {aez} — {set_name}",
                    save_path=os.path.join(
                        PLOT_SAVE_PATH,
                        f"{set_name.lower()}_true_vs_pred.png"
                    )
                )

            # ── Feature importance plot ───────────────────────
            plt.figure(figsize=(12, 5))
            importance = best_model.feature_importances_
            sns.barplot(
                x=importance, y=selected_feature_names,
                palette="viridis", hue=selected_feature_names, legend=False
            )
            plt.xlabel("Feature Importance Score")
            plt.ylabel("Features")
            plt.title(f"Feature Importance — {nutrient} — AEZ {aez}")
            plt.tight_layout()
            plt.savefig(
                os.path.join(PLOT_SAVE_PATH, "feature_importance.png"),
                dpi=150
            )
            plt.close()

            # ── Save model ────────────────────────────────────
            today     = str(datetime.datetime.today().date())
            file_name = f"rfr_model_{today}_AEZ_{aez}_{nutrient}.joblib"
            joblib.dump(best_model, os.path.join(OUTPUT_FOLDER, file_name))

            # ── Append to master stats ────────────────────────
            master_stats_list.append({
                "AEZ"         : aez,
                "Nutrient"    : nutrient,
                "CV_Mean_R2"  : round(float(cross_val_mean["R2"]),    4),
                "Test_R2"     : m_test["R2"],
                "Test_RMSE"   : m_test["RMSE"],
                "Test_MAE"    : m_test["MAE"],
                "Test_sMAPE"  : m_test["sMAPE"]
            })

            print(f"  ✓ Saved: {file_name}")

        except Exception as e:
            print(f"  ✗ FAILED — AEZ {aez} | {nutrient} → {e}")
            failed_list.append({"AEZ": aez, "Nutrient": nutrient, "Error": str(e)})
            continue

# ============================================================
# 5. Save Master Statistics
# ============================================================
master_df = pd.DataFrame(master_stats_list)
master_csv_path = os.path.join(STATS_FOLDER, "pan_india_model_metrics.csv")
master_df.to_csv(master_csv_path, index=False)

print(f"\n\n{'='*60}")
print(f"  TRAINING COMPLETE")
print(f"  Models trained : {len(master_stats_list)} / {total_models}")
print(f"  Master CSV     : {master_csv_path}")
print(f"{'='*60}")
print(master_df.to_string(index=False))

# ── Report any failures ───────────────────────────────────────
if failed_list:
    failed_df = pd.DataFrame(failed_list)
    failed_csv = os.path.join(STATS_FOLDER, "failed_models.csv")
    failed_df.to_csv(failed_csv, index=False)
    print(f"\n⚠  {len(failed_list)} model(s) failed — see {failed_csv}")
    print(failed_df.to_string(index=False))
else:
    print("\n✓  All models trained successfully. No failures.")


  PROCESSING AEZ: 3
  Loaded: 25299 rows, 53 columns

  [1/4]  AEZ 3  |  N
  ----------------------------------------
  Optuna best R²: 0.5284
  Params: split=2, leaf=3, features=0.599
    Fold  1  R²=0.5152  RMSE=73.5691  MAE=55.5711  sMAPE=29.3521
    Fold  2  R²=0.4919  RMSE=71.5647  MAE=53.7652  sMAPE=28.5879
    Fold  3  R²=0.4728  RMSE=71.3610  MAE=53.9004  sMAPE=28.4517
    Fold  4  R²=0.5404  RMSE=68.3903  MAE=52.3422  sMAPE=27.7939
    Fold  5  R²=0.5242  RMSE=68.8282  MAE=52.6880  sMAPE=27.5829
    Fold  6  R²=0.5991  RMSE=66.8677  MAE=50.5305  sMAPE=26.9890
    Fold  7  R²=0.5221  RMSE=70.9637  MAE=53.8406  sMAPE=28.0528
    Fold  8  R²=0.5037  RMSE=71.6174  MAE=54.4333  sMAPE=28.5451
    Fold  9  R²=0.5273  RMSE=71.9496  MAE=54.3188  sMAPE=28.4169
    Fold 10  R²=0.5070  RMSE=71.6219  MAE=54.6254  sMAPE=28.4815
  CV Mean R²: 0.5204
  Best fold: 6  R²=0.5991

  CAL_TRAIN  → R²=0.7888  RMSE=46.7935  MAE=35.134
  CAL_TEST   → R²=0.5991   RMSE=66.8677  MAE=50.5305
  TEST      